Load data from individual daily netcdf files, using tools by Justine Charrel, concatenate to single dataset and export to single netcdf file.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

#import personnal tools
import sys
sys.path.append('../tools/')
from info import *
from tools_generic import *

# Load files

In [ ]:
ds = open_single_dataset(
    prefix = 'FLOWCAPT',
    data_folder_path="/bdd/AWACA/TRANSECT/awacasurf",
    site='d17',
    year = '2025',
    month = '05',
    day = '19',
    drop_spc_bins=True,
    resample_mean=False)
ds

In [ ]:
data_folder_path='/bdd/AWACA/TRANSECT/awacasurf'
site_list=["d17","d47","d85","dmc"]
#site_list=['d17']
#site_list=["d47","d85","dmc"]


In [ ]:
prefix='SURF_TABLE'

begin_year="2024"
begin_month="12"
begin_day="01"
end_year="2025"
end_month="03"
end_day="13"

# begin_year="2024"
# begin_month="12"
# begin_day="01"
# end_year="2026"
# end_month="06"
# end_day="30"
begin_date_string = begin_year + begin_month + begin_day
end_date_string = end_year + end_month + end_day

In [ ]:
# for prefix in sensors:
if False:
    datasets = open_and_concatenate_datasets(
        prefix=prefix,
        data_folder_path="/bdd/AWACA/TRANSECT/awacasurf",
        site_list=site_list,
        begin_year=begin_year,
        begin_month=begin_month,
        begin_day=begin_day,
        end_year=end_year,
        end_month=end_month,
        end_day=end_day,
        resample_mean = True,
        reindex=True
    )

    #convert SPC snowflux for g/cm^2/s to g/m^2/s
    if prefix == 'SPC' :
        for site in site_list:
            if datasets[site] is not None:
                if 'snowflux' in datasets[site]:
                    print(f"Unit updated for snowflux at {site}")
                    datasets[site]['snowflux'] = datasets[site]['snowflux'] * 10000
                    datasets[site]['snowflux'].attrs['units'] = 'g/m^2/s'

    #export to single netcdf
    for site in site_list:
        ds=datasets.get(site)
        if ds is not None:
            #create netcdf file
            file_path=f'../../data/{prefix}_{site}_{begin_date_string}_{end_date_string}.netcdf'
            # Remove the file if it already exists
            if os.path.exists(file_path):
                os.remove(file_path)
                print(f"Existing file {file_path} removed.")
            ds.to_netcdf(file_path)

In [ ]:
wind_beginning=True # handle initial period where wind data stored differently

if wind_beginning:
    prefix='SURF_TABLE'

    begin_year="2024"
    begin_month="12"
    begin_day="01"
    end_year="2025"
    end_month="03"
    end_day="13"
        
    datasets = open_and_concatenate_datasets(
        prefix=prefix,
        data_folder_path="/bdd/AWACA/TRANSECT/awacasurf",
        site_list=site_list,
        begin_year=begin_year,
        begin_month=begin_month,
        begin_day=begin_day,
        end_year=end_year,
        end_month=end_month,
        end_day=end_day,
        resample_mean = True,
        reindex=True
    )

    #keep only relevant var
    var_list=["wspd1_Avg","wspd2_Avg","wspd3_Avg", "wdir"]
    rename_dict = { "wspd1_Avg":"wspd1",
                    "wspd2_Avg":"wspd2",
                    "wspd3_Avg":"wspd3"
                    }
    rename_dict_DMC = { "wspd1_Avg":"wspd1",
                        "wspd2_Avg":"wspd2"
                    }
    for site in site_list:
        ds=datasets.get(site)
        datasets[site]=ds.drop_vars([v for v in ds.data_vars if v not in var_list])
        if site=="dmc":
            datasets[site]=datasets[site].rename(rename_dict_DMC)
        else:
            datasets[site]=datasets[site].rename(rename_dict)
        

    #export to single netcdf
    for site in site_list:
        ds=datasets.get(site)
        if ds is not None:
            #create netcdf file
            file_path=f'../../data/WIND_BEGINNING_{site}_{begin_date_string}_{end_date_string}.netcdf'
            # Remove the file if it already exists
            if os.path.exists(file_path):
                os.remove(file_path)
                print(f"Existing file {file_path} removed.")
            ds.to_netcdf(file_path)


In [ ]:
datasets.get("d17")

In [ ]:
datasets.get("d47")

In [ ]:
datasets.get("d85")

In [ ]:
datasets.get("dmc")

# Basic plots

In [ ]:
ds=datasets.get("d17")
# ds=alias1
vars=['wspd1', 'wspd2', 'wspd3']
vars=['snowflux']
for var in vars:
    ds[var].plot()

# Export to unique netcdf

In [ ]:
for site in site_list:
    ds=datasets.get(site)
    if ds is not None:
        #create netcdf file
        file_path=f'../../data/{prefix}_{site}_{begin_date_string}_{end_date_string}.netcdf'
        # Remove the file if it already exists
        if os.path.exists(file_path):
            os.remove(file_path)
            print(f"Existing file {file_path} removed.")
        ds.to_netcdf(file_path)